# ระดับ 3g — Few-shot adaptation บน Celeb-DF (ทำซ้ำ protocol "adaptation" ของบทความ, Table V)

**บทความ (Sec. IV-C2):** นำโมเดลที่ฝึกบน FF++ แล้วมา fine-tune ต่อด้วยภาพจากชุดข้อมูลที่ไม่เคยเห็นเพียง **10 หรือ 20 ภาพ (real : fake เท่ากัน)**
**2 epoch, lr 2e-5** แล้ววัดผลทั้งบนชุดเป้าหมาย, บนชุดเดิม (FF++) และบนชุดอื่นที่ไม่เคยเห็น

**ความต่างจากบทความ (ต้องระบุในรายงาน):**
- ชุดเป้าหมายของบทความคือ DFDC / FFIW (เราไม่มีข้อมูล) → ใช้ **Celeb-DF** เป็นเป้าหมาย, **WildDeepfake** เป็นชุดที่ไม่เคยเห็น
- repo ไม่มีโค้ด few-shot และบทความไม่ระบุวิธีเลือกภาพ, batch size, optimizer, การทำซ้ำ → กำหนดเองดังนี้
  - เลือก **วิดีโอ k วิดีโอ (k/2 real + k/2 fake) วิดีโอละ 1 frame** แบบสุ่ม · สุ่ม **3 รอบ** (ชุดภาพเดียวกันสำหรับทุกโมเดล → เปรียบเทียบแบบ paired)
  - **กัน leakage:** ตัดวิดีโอที่ใช้ปรับ และทุกวิดีโอที่มี identity ร่วมกัน ออกจากชุดวัดผล Celeb-DF · AUC "ก่อน" และ "หลัง" วัดบนชุดที่เหลือชุดเดียวกัน
  - fine-tune **ทั้งโมเดล** ด้วย loss เดิมของแต่ละ variant (FB·CE: L_ce · CFB full: Eq. 9) · augmentation เดิม · Lamb, lr คงที่ 2e-5, batch 8 · inference ใช้ main branch
- โมเดล: checkpoint FB·CE และ CFB full จากระดับ 3d2 (seed 2025, 3407)

**Add Input:** output ของ **3d2** (`ablation_seeds/`) · **3a** (FF++ test) · **3c** (`cdf_crops`) · 3e ที่มี `wdf_crops/` · GPU **T4 ×1** · Internet On
**เวลา:** ประมาณ 40–60 นาที

## 0. ตั้งค่า

In [ ]:
import os, sys, re, json, time, glob, random, shutil, tarfile, hashlib, subprocess, warnings, io
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # ไม่พิมพ์ progress bar ของการดาวน์โหลด 157 ไฟล์ (ทำให้ notebook ใหญ่เกิน 1 MB ตอน commit)
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR   = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR  = TMP_DIR / "FFDBackbone"
DATA_ROOT = TMP_DIR / "data"                     # แตก tar ของระดับ 3a (เฉพาะ test)
WDF_DIR   = WORK_DIR / "wdf_crops"               # เก็บไว้ใน working เพื่อใช้ซ้ำรอบหน้า (Add Input)
WEIGHT_DIR = TMP_DIR / "weights"
OUT_DIR   = WORK_DIR / "results_level3g"
for d in [TMP_DIR, DATA_ROOT, WDF_DIR, WEIGHT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1jFIpb4TftJiL82h69c3Roqy8uIzGQP4G"

# --- โมเดล ---
USE_AUTHOR_CKPT   = False                         # ดาวน์โหลด checkpoint "Ours" ของผู้เขียนมาเทียบ
AUTHOR_CKPT_PATH  = None                         # ถ้าอัปเป็น Kaggle Dataset ไว้แล้วให้ใส่ path
AUTHOR_HINTS      = ["ours", "final", "ddbf", "dbbf", "mainbranch"]
AUTHOR_EXCLUDE    = ["beit-1k-face-55w"]

# --- dataset ---
DATASETS_P2 = [d for d in os.environ.get("FFD_DATASETS", "FF++,CDF,WDF").split(",") if d]  # ชุดที่ใช้ในส่วน 2
WDF_REPO   = "xingjunm/WildDeepfake"
WDF_SPLITS = {"real_test": 0, "fake_test": 1}    # label: 0 = real, 1 = fake
WDF_REALIGN = True                               # True = RetinaFace + align ซ้ำแบบเดียวกับ FF++/CDF (หาไม่เจอ → ใช้ภาพเดิม)
WDF_MAX_FILES = None                             # จำกัดจำนวนไฟล์ tar ต่อ split (None = ทั้งหมด) — ใช้ทดสอบเร็ว ๆ

# --- preprocessing / inference (ค่าตาม repo) ---
N_FRAMES, IMG_SIZE = 8, 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
DET_THRES = 0.5
BATCH_SIZE, NUM_WORKERS, USE_AMP = 64, 2, True

# --- UTOM (Sec. III-D + Supplementary 2.1, 2.3) ---
UTOM_LAMBDA, UTOM_EPS = 0.02, 1e-7
N_BOOT, SEED = 1000, 42

PAPER_MAIN = {"FF++": 99.36, "CDF": 90.46}
PAPER_TABLE6 = {   # Table VI: CDF, DFDC, FFIW, Avg AUC, Avg ACC
    "fb_ce": (89.18, 84.02, 86.48, 86.56, 79.37), "fb_edu": (89.34, 84.19, 88.76, 87.43, 80.96),
    "cfb_base": (89.77, 83.67, 87.69, 87.04, 79.26), "cfb_dec": (90.31, 84.89, 89.21, 88.14, 79.42),
    "cfb_dec_ufm": (91.16, 84.54, 90.22, 88.64, 80.84), "cfb_full": (90.46, 84.90, 90.97, 88.78, 82.29),
}
PAPER_TABLE9_CDF = {   # Table IX, CFB + L_EDU, source = CDF → target CDF (ACC %)
    "tau=0.5": 83.20, "tau_ot": 83.78, "tau_ot^UTOM": 85.33}

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE)


# --- few-shot (บทความ Sec. IV-C2: 10/20-shot, real:fake เท่ากัน, 2 epoch, lr 2e-5) ---
SHOTS     = [int(x) for x in os.environ.get("FFD_SHOTS", "10,20").split(",")]
N_DRAWS   = int(os.environ.get("FFD_DRAWS", "3"))      # สุ่มชุดภาพกี่รอบ
FT_EPOCHS = 2
FT_LR     = 2e-5
FT_BS     = int(os.environ.get("FFD_FT_BS", "8"))   # batch ต่อ step (T4 16 GB รับ 8 ได้สบาย)
FT_WD     = 0.05
FT_CLIP   = 1.0
FT_AMP    = False   # fp32 ตอน fine-tune: มีแค่ 4–6 step · AMP GradScaler (scale เริ่ม 65536) ข้าม step ที่ grad overflow ทำให้ weight ไม่เปลี่ยน
FS_VARIANTS = ["fb_ce", "cfb_full"]
EDL_CFG = dict(num_classes=2, evidence="exp", loss_type="log",
               with_kldiv=False, with_avuloss=True, annealing_method="exp")   # = DBBF.yaml
PAPER_TABLE5 = {"DFDC 10-shot": (84.90, 87.72), "DFDC 20-shot": (84.90, 88.07), "FFIW 20-shot": (90.97, 96.65)}

SEED_CKPTS = sorted(p for p in (INPUT_DIR.rglob("ablation_seeds/*_s*/best.pth") if INPUT_DIR.exists() else [])
                    if p.parent.name.rsplit("_s", 1)[0] in FS_VARIANTS)
_seen = {}
for p in SEED_CKPTS: _seen.setdefault(p.parent.name, p)          # Add Input ซ้ำหลายชุด → ใช้ตัวแรก
SEED_CKPTS = list(_seen.values())
assert SEED_CKPTS, "ไม่พบ ablation_seeds/fb_ce_s*/best.pth หรือ cfb_full_s*/best.pth — Add Input เป็น output ของระดับ 3d2"
DATASETS = ["FF++", "CDF", "WDF"]
print("checkpoint:", [p.parent.name for p in SEED_CKPTS], "| shots:", SHOTS, "| draws:", N_DRAWS)

## 1. ติดตั้ง + clone repo

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)
pip_install("gdown")
try: import huggingface_hub
except ImportError: pip_install("huggingface_hub")
try: import timm
except ImportError: pip_install("timm"); import timm

import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})
assert DEVICE.type == "cuda", ("ไม่พบ GPU — notebook นี้บน CPU จะใช้เวลา ~15–25 ชม. "
    "ตั้ง Notebook options → Accelerator = GPU T4 x1 (และเช็กโควตา GPU) แล้วรันใหม่")

def find_repo_in_input():
    """หา FFDBackbone ที่ Add Input ไว้ (โหมดออฟไลน์) — ค้นแค่ 2 ชั้นแรก ไม่ไล่ทั้ง dataset"""
    if not INPUT_DIR.exists():
        return None
    marker = Path("models") / "lib" / "BEiT_v2" / "modeling_finetune.py"
    for d in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
        cands = [d] + sorted(p for p in d.iterdir() if p.is_dir())[:50]
        for c in cands:
            if (c / marker).exists():
                return c
    return None

def get_repo():
    """1) ที่ clone ไว้แล้ว  2) จาก /kaggle/input (ออฟไลน์)  3) git clone"""
    if (REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py").exists():
        return "cached"
    src = find_repo_in_input()
    if src is not None:
        print("พบ repo ใน Add Input (โหมดออฟไลน์):", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
        return "input"
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone repo ไม่สำเร็จ:\n  " + (r.stderr or "").strip() + "\n\n"
            "ถ้าขึ้น 'Could not resolve host' แปลว่า Internet ปิดอยู่ — แก้ได้ 2 ทาง\n"
            "  (ก) เปิดเน็ต: แผงขวาของ Kaggle -> Notebook options -> Internet = On\n"
            "      (ต้องยืนยันเบอร์โทรที่ kaggle.com/settings ก่อน ถ้ายังไม่เคยทำ)\n"
            "  (ข) โหมดออฟไลน์: ดาวน์โหลด zip ของ repo จาก\n"
            "      https://github.com/zhenglab/FFDBackbone/archive/refs/heads/main.zip\n"
            "      -> อัปเป็น Kaggle Dataset -> Add Input  (cell นี้จะหาเจอเอง)")
    return "clone"

REPO_SOURCE = get_repo()
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "(จาก Add Input — ไม่มี .git)"

def patch_repo(repo: Path):
    n = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8"); n += 1
    return n

print("repo commit:", REPO_COMMIT, "| patched:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))

import importlib.util
def load_module_from_file(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

modeling_finetune = load_module_from_file("beitv2_modeling_finetune",
                                          REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
from face_process.lib.ct.detection import FaceDetector
from face_process import face_utils
from scipy.stats import rankdata, norm
pip_install("albumentations")
import albumentations as alb
from albumentations.pytorch.transforms import ToTensorV2
import timm.optim as topt
EDL = load_module_from_file("ffd_edl", REPO_DIR / "common" / "losses" / "edl_loss.py")   # L_EDU ของผู้เขียน

## 2. โมเดล (โครงสร้างเดียวกับระดับ 3e/3f) + forward สำหรับการฝึก

In [ ]:
BEIT_CONFIG = dict(drop_path_rate=0.1, use_mean_pooling=False, init_values=0.1, qkv_bias=True,
                   use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False)

def make_beit():
    m = modeling_finetune.beit_base_patch16_224(**BEIT_CONFIG)
    m.head = nn.Identity()
    return m

def evidential_uncertainty(logits, K=2):
    alpha = torch.exp(torch.clamp(logits, -10, 10)) + 1
    return K / alpha.sum(dim=1, keepdim=True)

class Detector(nn.Module):
    def __init__(self, dual: bool, ufm: bool = True):
        super().__init__()
        self.dual, self.ufm = dual, ufm
        self.backbone_model = make_beit()
        if dual: self.backbone_model2 = make_beit()
        self.norm_for_cls = nn.LayerNorm(768)
        self.header = nn.Sequential(nn.Linear(768, 2))
    def classify(self, f): return self.header(self.norm_for_cls(f))
    def forward(self, x):
        f1 = self.backbone_model(x)
        out = {"main": self.classify(f1)}
        if self.dual:
            f2 = self.backbone_model2(x)
            l2 = self.classify(f2)
            if self.ufm:
                u = torch.cat([evidential_uncertainty(out["main"].float()),
                               evidential_uncertainty(l2.float())], dim=-1)
                w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
                fused = w[:, :1] * f1 + w[:, 1:] * f2
            else:
                fused = f1 + f2
            out["branch2"] = l2
            out["fused"] = self.classify(fused)
        return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = {k: v for k, v in ck.items() if not isinstance(v, (dict, torch.Tensor))} if isinstance(ck, dict) else {}
    flags = ck.get("flags") if isinstance(ck, dict) else None
    if flags is None and isinstance(ck, dict) and isinstance(ck.get("cfg"), dict):
        flags = ck["cfg"].get("flags")
    sd = ck
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if isinstance(sd, dict) and isinstance(sd.get(key), dict):
            sd = sd[key]; break
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items() if torch.is_tensor(v)}
    return sd, meta, flags

def load_detector(path):
    sd, meta, flags = load_state_dict_any(path)
    dual = any(k.startswith("backbone_model2.") for k in sd)
    ufm = True if flags is None else bool(flags.get("ufm", True))
    m = Detector(dual=dual, ufm=ufm)
    expected = set(m.state_dict())
    loadable = {k: v for k, v in sd.items() if k in expected}
    missing = sorted(expected - set(loadable))
    bad = [k for k, v in loadable.items() if v.shape != m.state_dict()[k].shape]
    if missing or bad:
        raise RuntimeError(f"{path}: checkpoint ไม่ตรงกับโมเดล missing={missing[:5]} shape={bad[:5]}")
    m.load_state_dict(loadable, strict=True)
    info = {"dual": dual, "ufm": ufm, "flags": flags, "epoch": meta.get("epoch"), "variant": meta.get("variant")}
    return m.eval().to(DEVICE), info


MODELS = [(p.parent.name, p) for p in SEED_CKPTS]
for n, p in MODELS:
    print(f"  {n:<24} {p}  ({p.stat().st_size/1e6:.0f} MB)")

def pearson_correlation_loss(x, y):          # models/custom.py ของ repo (เหมือน 3b/3d2)
    xm = x.sub(x.mean(-1, keepdim=True)); ym = y.sub(y.mean(-1, keepdim=True))
    cov = torch.bmm(xm.unsqueeze(1), ym.unsqueeze(2)).squeeze() / x.size(-1)
    r = (cov / (x.std(-1) * y.std(-1))).mean()
    return torch.clamp(r, -1.0, 1.0)

def train_forward(model, x):
    """คืน logits ของ fused / main / aux และ Pearson loss · ตรงกับ DDBF_BEiT_v2.forward ของ 3b/3d2"""
    f1 = model.backbone_model(x)
    p1 = model.classify(f1)
    if not model.dual:
        return p1, p1, None, f1.new_zeros(())
    f2 = model.backbone_model2(x)
    ploss = pearson_correlation_loss(f1.detach().float(), f2.float())   # stop-gradient ที่ main
    p2 = model.classify(f2)
    if model.ufm:
        u = torch.cat([evidential_uncertainty(p1.float()), evidential_uncertainty(p2.float())], dim=-1)
        w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
        fused = w[:, :1] * f1 + w[:, 1:] * f2
    else:
        fused = f1 + f2
    return model.classify(fused), p1, p2, ploss

## 3. ชุดข้อมูล (FF++ test จาก 3a · Celeb-DF จาก 3c · WildDeepfake จาก 3e)

In [ ]:
# ---------- FF++ test (จากระดับ 3a) ----------
FF_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
REAL_REL = "original_sequences/youtube/c23/videos"

def frames_of(d, k=N_FRAMES):
    """เหมือน get_sampled_idx() ของ repo: step = N // k แล้วเอา k ตัวแรก"""
    if not d.is_dir(): return []
    idxs = sorted(int(p.stem) for p in d.glob("*.png") if p.stem.isdigit())
    if not idxs: return []
    sel = idxs[::len(idxs) // k][:k] if len(idxs) >= k else sorted(np.resize(np.array(idxs), k).tolist())
    return [str(d / f"{i}.png") for i in sorted(sel)]

def build_ffpp_test():
    tars = sorted(INPUT_DIR.rglob("*.tar")) if INPUT_DIR.exists() else []
    tars = {t.stem: t for t in tars if t.stem in ["meta", "original"] + FF_METHODS}
    face_root = None
    if "meta" in tars:
        with tarfile.open(tars["meta"]) as tf: tf.extractall(DATA_ROOT)
        face_root = DATA_ROOT / "FF++" / "face_v1"
    else:
        hits = sorted(INPUT_DIR.rglob("face_v1/splits/test.json")) if INPUT_DIR.exists() else []
        if hits: face_root = hits[0].parent.parent
    if face_root is None or not (face_root / "splits" / "test.json").exists():
        raise RuntimeError("ไม่พบข้อมูล FF++ จากระดับ 3a — Add Input เป็น output ของ notebook FFD_level3a")
    pairs = json.load(open(face_root / "splits" / "test.json"))
    rows = []
    for t, s in pairs:
        rows += [{"rel": f"{REAL_REL}/{x}", "label": 0, "method": "original", "video": x} for x in (t, s)]
        for m in FF_METHODS:
            rows += [{"rel": f"manipulated_sequences/{m}/c23/videos/{x}", "label": 1, "method": m, "video": x}
                     for x in (f"{t}_{s}", f"{s}_{t}")]
    # แตกเฉพาะไฟล์ของ test split จาก tar (ประหยัดเวลา/พื้นที่)
    want = tuple(f"FF++/face_v1/{r['rel']}/" for r in rows)
    for stem in ["original"] + FF_METHODS:
        if stem not in tars: continue
        marker = DATA_ROOT / f".test_{stem}"
        if marker.exists(): continue
        with tarfile.open(tars[stem]) as tf:
            tf.extractall(DATA_ROOT, members=[m for m in tf.getmembers() if m.name.startswith(want)])
        marker.touch()
    out = []
    for r in rows:
        fr = frames_of(face_root / r["rel"] / "frame")
        if len(fr) == N_FRAMES:
            out.append({"uid": f"{r['method']}/{r['video']}", "label": r["label"], "group": r["method"], "frames": fr})
    df = pd.DataFrame(out)
    print(f"FF++ test: {len(df)} วิดีโอ (คาด 700) · real {int((df.label==0).sum())} / fake {int((df.label==1).sum())}")
    return df

# ---------- Celeb-DF (crop จากระดับ 3c) ----------
CDF_CROP_DIR = WORK_DIR / "cdf_crops"                # ใช้เมื่อต้อง crop เอง (output ของ 3c ไม่ครบ)

def read_crop_root(root):
    out = []
    for d in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        subset = d.name.split("__")[0]
        fr = sorted(d.glob("*.png"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
        if not fr: continue
        fr = [str(p) for p in (fr[:N_FRAMES] if len(fr) >= N_FRAMES else np.resize(np.array(fr), N_FRAMES))]
        out.append({"uid": d.name, "label": int("synthesis" in subset.lower()), "group": subset, "frames": fr})
    return pd.DataFrame(out, columns=["uid", "label", "group", "frames"])

def build_cdf():
    """อ่าน cdf_crops จาก output ของ 3c · ถ้าไม่ครบ 518 คืนค่าที่อ่านได้ แล้วให้ cell ถัดไป crop เองจาก dataset ดิบ"""
    roots = sorted(p for p in INPUT_DIR.rglob("cdf_crops") if p.is_dir()) if INPUT_DIR.exists() else []
    if CDF_CROP_DIR.exists(): roots.append(CDF_CROP_DIR)
    best = None
    for r in roots:
        df = read_crop_root(r)
        print(f"  cdf_crops: {len(df):4d} วิดีโอ ← {r}")
        if best is None or len(df) > len(best[0]): best = (df, r)
    if best is None:
        print("  ไม่พบ cdf_crops ใน Add Input"); return pd.DataFrame(columns=["uid", "label", "group", "frames"])
    df, root = best
    print(f"Celeb-DF: {len(df)} วิดีโอ (คาด 518) · real {int((df.label==0).sum())} / fake {int((df.label==1).sum())}  ← {root}")
    return df

DATA = {}
if "FF++" in DATASETS: DATA["FF++"] = build_ffpp_test()
if "CDF"  in DATASETS: DATA["CDF"]  = build_cdf()

In [ ]:
# ---------- face detector + align (เหมือนระดับ 3c) ----------
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def crop_face_test(img, bbox):
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w / 4 * 0.5; h0 = h / 4 * 0.5
    return img[max(0, int(y0-h0)):min(H, int(y1+h0)+1), max(0, int(x0-w0)):min(W, int(x1+w0)+1)]

def realign(img_bgr):
    """คืน (ภาพ 224x224, aligned?) — ขยายภาพก่อนตรวจ เพราะภาพ WDF เป็นหน้าที่ crop มาแล้วขนาดเล็ก"""
    big = cv2.resize(img_bgr, (448, 448), interpolation=cv2.INTER_LINEAR)
    try:
        faces = [f for f in detector.detect(big) if f[2] >= DET_THRES]
    except Exception:
        faces = []
    if faces:
        box, lm, _ = max(faces, key=lambda f: (f[0][2]-f[0][0]) * (f[0][3]-f[0][1]))
        try:
            aligned, _, bb = face_utils.crop_aligned(big, lm.astype(np.float64), landmarks_68=None,
                                                    bboxes=box.astype(np.float64), aligned_image_size=IMG_SIZE)
            crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
            if crop.size:
                return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR), True
        except Exception:
            pass
    return cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR), False

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")
num_key = lambda n: (int(re.sub(r"\D", "", Path(n).stem) or 0), n)

def pick_k(names, k=N_FRAMES):
    names = sorted(names, key=num_key)
    if len(names) >= k:
        return names[::len(names) // k][:k]
    return sorted(np.resize(np.array(names), k).tolist(), key=num_key)

def build_wdf():
    meta_csv = WDF_DIR / "wdf_meta.csv"
    prev = sorted(INPUT_DIR.rglob("wdf_crops/wdf_meta.csv")) if INPUT_DIR.exists() else []
    if prev and not meta_csv.exists():
        print("ใช้ wdf_crops จาก Add Input:", prev[0].parent)
        m = pd.read_csv(prev[0]); root = prev[0].parent
        m["frames"] = m["frames"].map(lambda s: [str(root / x) for x in json.loads(s)])
        need = [p_ for fs in m["frames"] for p_ in fs]
        miss = sum(not Path(p_).exists() for p_ in need)
        if miss == 0:
            return m
        print(f"  wdf_crops จาก Add Input ไม่ครบ (ขาด {miss}/{len(need)} ภาพ) → สร้างใหม่จาก HuggingFace (ต้องเปิด Internet)")
    from huggingface_hub import HfApi, hf_hub_download
    files = HfApi().list_repo_files(WDF_REPO, repo_type="dataset")
    todo = []
    for split, lab in WDF_SPLITS.items():
        fs = sorted([f for f in files if f"/{split}/" in f and f.endswith(".tar.gz")], key=num_key)
        todo += [(f, split, lab) for f in fs[:WDF_MAX_FILES]]
    print(f"WildDeepfake: {len(todo)} ไฟล์ tar · " +
          ", ".join(f"{s} {sum(t[1]==s for t in todo)}" for s in WDF_SPLITS))
    rows, n_aligned, n_img, shown = [], 0, 0, False
    dl_dir = TMP_DIR / "wdf_dl"; dl_dir.mkdir(parents=True, exist_ok=True)
    for f, split, lab in tqdm(todo, desc="WDF"):
        local = Path(hf_hub_download(WDF_REPO, f, repo_type="dataset", local_dir=str(dl_dir)))
        vid = Path(f).name.replace(".tar.gz", "")
        with tarfile.open(local, "r:*") as tf:
            names = [m.name for m in tf.getmembers() if m.isfile() and m.name.lower().endswith(IMG_EXT)]
        seqs = {}
        for n in names: seqs.setdefault(str(Path(n).parent), []).append(n)
        if not shown:
            print(f"\nตัวอย่างโครงสร้าง {f}: {len(names)} ภาพ, {len(seqs)} sequence")
            for s_, ns in list(seqs.items())[:3]: print("  ", s_, "→", len(ns), "ภาพ เช่น", ns[:2])
            shown = True
        want = {}
        for s_, ns in seqs.items():
            for j, n in enumerate(pick_k(ns)): want.setdefault(n, []).append((s_, j))
        imgs = {}
        with tarfile.open(local, "r|*") as tf:               # อ่านรอบเดียวแบบ stream (r|* = ตรวจเองว่าบีบอัดหรือไม่ — ไฟล์ .tar.gz ของ WDF เป็น tar ธรรมดา)
            for m in tf:
                if m.name in want:
                    buf = np.frombuffer(tf.extractfile(m).read(), np.uint8)
                    imgs[m.name] = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        for s_, ns in seqs.items():
            sid = re.sub(r"[^\w.-]", "_", f"{vid}__{s_}")
            out_dir = WDF_DIR / split / sid; out_dir.mkdir(parents=True, exist_ok=True)
            rel = []
            for j, n in enumerate(pick_k(ns)):
                im = imgs.get(n)
                if im is None: continue
                if WDF_REALIGN:
                    im, ok = realign(im); n_aligned += ok
                else:
                    im = cv2.resize(im, (IMG_SIZE, IMG_SIZE))
                n_img += 1
                p = out_dir / f"{j}.png"; cv2.imwrite(str(p), im); rel.append(str(p.relative_to(WDF_DIR)))
            if rel:
                rel = (rel * N_FRAMES)[:N_FRAMES]
                rows.append({"uid": f"{split}/{sid}", "label": lab, "group": split, "source_video": vid,
                             "n_frames_seq": len(ns), "frames": json.dumps(rel)})
        local.unlink(missing_ok=True)
    m = pd.DataFrame(rows)
    m.to_csv(meta_csv, index=False)
    print(f"\nrealign สำเร็จ {n_aligned}/{n_img} ภาพ ({100*n_aligned/max(n_img,1):.1f}%) — ที่เหลือใช้ภาพเดิม resize เป็น 224")
    m["frames"] = m["frames"].map(lambda s: [str(WDF_DIR / x) for x in json.loads(s)])
    return m

if "WDF" in DATASETS:
    t0 = time.time()
    DATA["WDF"] = build_wdf()
    w = DATA["WDF"]
    print(f"WildDeepfake: {len(w)} sequence จาก {w['source_video'].nunique()} วิดีโอ · "
          f"real {int((w.label==0).sum())} / fake {int((w.label==1).sum())} · {(time.time()-t0)/60:.1f} นาที")

# ---------- Celeb-DF: ถ้า cdf_crops จาก 3c ไม่ครบ → crop เองจาก dataset ดิบ (โค้ดเดียวกับระดับ 3c) ----------
MAX_SHIFT = 10

def _valid_faces(faces, thres=DET_THRES):
    return [(b.astype(np.float64), l.astype(np.float64), float(s)) for b, l, s in faces[:10] if s >= thres]

def _box_iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1]); x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x1 - x0) * max(0., y1 - y0)
    area = lambda r: max(0., r[2] - r[0]) * max(0., r[3] - r[1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def _crop_video(path, out_dir):
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0: cap.release(); return 0
    out_dir.mkdir(parents=True, exist_ok=True)
    idx = np.arange(n); targets = idx[::max(1, n // N_FRAMES)][:N_FRAMES] if n >= N_FRAMES else np.sort(np.resize(idx, N_FRAMES))
    prev, saved = None, 0
    for t in targets:
        for sh in range(MAX_SHIFT + 1):
            fi = int(t) + sh
            if fi >= n: break
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, frame = cap.read()
            if not ok or frame is None: continue
            try: faces = _valid_faces(detector.detect(frame))
            except Exception: continue
            if not faces: continue
            face = (max(faces, key=lambda f: (f[0][2]-f[0][0]) * (f[0][3]-f[0][1])) if prev is None
                    else max(faces, key=lambda f: _box_iou(f[0], prev)))
            try:
                aligned, _, bb = face_utils.crop_aligned(frame, face[1], landmarks_68=None, bboxes=face[0],
                                                        aligned_image_size=IMG_SIZE)
                crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
            except Exception:
                continue
            if crop.size == 0: continue
            cv2.imwrite(str(out_dir / f"{fi}.png"), cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR))
            prev, saved = face[0], saved + 1
            break
    cap.release(); return saved

if "CDF" in DATASETS and len(DATA.get("CDF", [])) < 518:
    lists = sorted(INPUT_DIR.rglob("List_of_testing_videos.txt")) if INPUT_DIR.exists() else []
    if not lists:
        raise RuntimeError("cdf_crops จาก 3c ไม่ครบ และไม่พบ Celeb-DF ดิบ — Add Input dataset 'celebdf-v2-test518' "
                           "(ชุดเดียวกับที่ระดับ 3c ใช้) แล้วรันใหม่")
    cdf_root = lists[0].parent
    print(f"\ncdf_crops ไม่ครบ ({len(DATA.get('CDF', []))}/518) → crop เองจาก {cdf_root} (~6–7 นาทีบน T4)")
    t0 = time.time()
    rows_ = [l.split() for l in (cdf_root / "List_of_testing_videos.txt").read_text().split("\n") if len(l.split()) == 2]
    for _, rel in tqdm(rows_, desc="crop Celeb-DF", mininterval=30):
        rel = rel.replace("\\", "/")
        sid = rel[:-4].replace("/", "__")
        od = CDF_CROP_DIR / sid
        if od.exists() and len(list(od.glob("*.png"))) >= N_FRAMES: continue
        _crop_video(cdf_root / rel, od)
    DATA["CDF"] = read_crop_root(CDF_CROP_DIR)
    print(f"crop เสร็จใน {(time.time()-t0)/60:.1f} นาที")

if "CDF" in DATASETS:
    c_ = DATA["CDF"]
    print(f"Celeb-DF ที่ใช้: {len(c_)} วิดีโอ · real {int((c_.label==0).sum())} / fake {int((c_.label==1).sum())}")
    if len(c_) < 518 or c_.label.nunique() < 2:
        raise RuntimeError(f"Celeb-DF ไม่ครบ ({len(c_)}/518) — ตรวจ Add Input")


## 4. Inference → score ระดับวิดีโอ

In [ ]:
def to_tensor(rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

class FrameList(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        rgb = np.asarray(Image.open(self.paths[i]).convert("RGB"))
        if rgb.shape[:2] != (IMG_SIZE, IMG_SIZE):
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return to_tensor(rgb)

@torch.no_grad()
def score_dataset(model, df):
    paths = [p for fs in df["frames"] for p in fs]
    owner = np.repeat(np.arange(len(df)), [len(fs) for fs in df["frames"]])
    loader = DataLoader(FrameList(paths), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    acc = {}
    for x in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
            out = model(x)
        for h, lg in out.items():
            lg = lg.float()
            acc.setdefault(f"p_{h}", []).append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
            acc.setdefault(f"u_{h}", []).append(evidential_uncertainty(lg)[:, 0].cpu().numpy())
    fr = pd.DataFrame({k: np.concatenate(v) for k, v in acc.items()}); fr["row"] = owner
    vid = fr.groupby("row").mean()                                  # เฉลี่ย 8 frame = score วิดีโอ
    return pd.concat([df[["uid", "label", "group"]].reset_index(drop=True), vid.reset_index(drop=True)], axis=1)

## 5. Few-shot: เลือกภาพ · กัน identity leakage · fine-tune

ชื่อวิดีโอ Celeb-DF: `Celeb-real/id0_0000` (identity id0) · `Celeb-synthesis/id0_id16_0000` (id0 และ id16) · `YouTube-real/00000` (ไม่ใช่ identity ใน Celeb)

In [ ]:
def cdf_ids(uid):
    sub, stem = uid.split("__", 1) if "__" in uid else ("", uid)
    ids = set(re.findall(r"id\d+", stem))
    return ids if ids else {f"{sub}:{stem}"}

CDF_IDS = {u: cdf_ids(u) for u in DATA["CDF"].uid}

def draw_shots(k, draw):
    """k วิดีโอ (k/2 real + k/2 fake) วิดีโอละ 1 frame · ชุดเดียวกันสำหรับทุกโมเดล"""
    rng = np.random.default_rng(10_000 * k + draw)
    df = DATA["CDF"]
    pick = []
    for lab in (0, 1):
        pool = df[df.label == lab]
        rows = pool.iloc[rng.choice(len(pool), k // 2, replace=False)]
        for _, r_ in rows.iterrows():
            pick.append((r_.uid, r_.frames[rng.integers(len(r_.frames))], int(lab)))
    used = set().union(*[CDF_IDS[u] for u, _, _ in pick])
    rest = df[[not (CDF_IDS[u] & used) for u in df.uid]].reset_index(drop=True)
    return pick, rest, sorted(used)

TF_TRAIN = alb.Compose([
    alb.RGBShift((-20, 20), (-20, 20), (-20, 20), p=0.3),
    alb.HueSaturationValue(hue_shift_limit=(-0.3, 0.3), sat_shift_limit=(-0.3, 0.3), val_shift_limit=(-0.3, 0.3), p=0.3),
    alb.RandomBrightnessContrast(brightness_limit=(-0.3, 0.3), contrast_limit=(-0.3, 0.3), p=0.3),
    (alb.ImageCompression(quality_range=(40, 100), p=0.5) if "quality_range" in alb.ImageCompression.__init__.__code__.co_varnames
     else alb.ImageCompression(quality_lower=40, quality_upper=100, p=0.5)),
    alb.HorizontalFlip(), alb.ToGray(p=0.01), alb.Resize(IMG_SIZE, IMG_SIZE),
    alb.Normalize(mean=MEAN, std=STD), ToTensorV2()])                    # = create_base_transforms(train) ของ repo

def cutout(img):                                                         # = map_util.Cutout ของ repo
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    length = int(h * np.random.randint(1, 30) * 0.01) * 2
    for _ in range(np.random.randint(1, 4)):
        y, x = np.random.randint(h), np.random.randint(w)
        mask[np.clip(y - length // 2, 0, h):np.clip(y + length // 2, 0, h),
             np.clip(x - length // 2, 0, w):np.clip(x + length // 2, 0, w)] = 0.
    return img * torch.from_numpy(mask)

def load_shot(path):
    rgb = np.asarray(Image.open(path).convert("RGB"))
    return cutout(TF_TRAIN(image=rgb)["image"])

def finetune(model, info, pick, seed):
    """fine-tune ทั้งโมเดลด้วย loss เดิมของ variant · คืนค่า loss ราย step"""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    flags = info["flags"] or {"dual": info["dual"], "dec": True, "ufm": info["ufm"], "edu": True}
    model.train()
    opt = topt.Lamb([p for p in model.parameters() if p.requires_grad], lr=FT_LR, weight_decay=FT_WD)
    use_amp = FT_AMP and DEVICE.type == "cuda"
    try: scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError): scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ce = nn.CrossEntropyLoss(); edl = EDL.EvidenceLoss(**EDL_CFG).to(DEVICE)
    def edu(o, y, ep):
        d = edl(o, y, epoch=ep, total_epoch=FT_EPOCHS)
        return sum(v.mean() for k_, v in d.items() if "loss" in k_ and torch.is_tensor(v))
    losses = []
    for ep in range(1, FT_EPOCHS + 1):
        order = np.random.permutation(len(pick))
        for b in range(0, len(order), FT_BS):
            idx = order[b:b + FT_BS]
            x = torch.stack([load_shot(pick[j][1]) for j in idx]).to(DEVICE)
            y = torch.tensor([pick[j][2] for j in idx], device=DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                pf, pm, pa, ploss = train_forward(model, x)
            pf, pm = pf.float(), pm.float()
            if info["dual"]:
                branch = (lambda o: edu(o, y, ep)) if flags.get("edu", True) else (lambda o: ce(o, y))
                loss = ce(pf, y) + branch(pm) + branch(pa.float()) + (ploss.float() if flags.get("dec", True) else 0.0)
            else:
                loss = edu(pm, y, ep) if flags.get("edu", False) else ce(pm, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), FT_CLIP)
            scaler.step(opt); scaler.update()
            losses.append(float(loss))
    model.eval()
    return losses

def auc(v): return 100 * roc_auc_score(v.label, v.p_main)

## 6. รัน: baseline (ก่อนปรับ) → fine-tune → วัดผลหลังปรับ

In [ ]:
ROWS, BASE = [], {}
SHOT_LOG = []
for k in SHOTS:
    for d in range(N_DRAWS):
        pick, rest, used = draw_shots(k, d)
        SHOT_LOG.append({"k": k, "draw": d, "shots": [u for u, _, _ in pick], "excluded_ids": used,
                         "cdf_rest_n": len(rest), "cdf_rest_real": int((rest.label == 0).sum()),
                         "cdf_rest_fake": int((rest.label == 1).sum())})
json.dump(SHOT_LOG, open(OUT_DIR / "fewshot_draws.json", "w"), indent=1, ensure_ascii=False)
print(pd.DataFrame(SHOT_LOG)[["k", "draw", "cdf_rest_n", "cdf_rest_real", "cdf_rest_fake"]].to_string(index=False))

for name, path in MODELS:
    t0 = time.time()
    model, info = load_detector(path)
    base_sd = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
    BASE[name] = {ds: score_dataset(model, df) for ds, df in DATA.items()}
    for ds, v in BASE[name].items(): v.to_csv(OUT_DIR / f"scores_{name}_base_{ds}.csv", index=False)
    print(f"\n{name}: baseline " + " · ".join(f"{ds} {auc(v):.2f}" for ds, v in BASE[name].items()))
    for k in SHOTS:
        for d in range(N_DRAWS):
            pick, rest, used = draw_shots(k, d)
            model.load_state_dict(base_sd)
            losses = finetune(model, info, pick, seed=1000 * k + d)
            if k == SHOTS[0] and d == 0:                     # ตรวจว่า weight เปลี่ยนจริง (ทุก branch)
                sd_now = model.state_dict()
                for key in ["header.0.weight", "backbone_model.blocks.11.mlp.fc2.weight", "backbone_model2.blocks.11.mlp.fc2.weight"]:
                    if key in sd_now:
                        rel = ((sd_now[key].float().cpu() - base_sd[key].float()).norm() / base_sd[key].float().norm()).item()
                        print(f"    weight เปลี่ยน {key}: {rel:.2e}")
                        assert rel > 0, f"{key} ไม่เปลี่ยนหลัง fine-tune — optimizer ไม่ได้ step (ตรวจ AMP/GradScaler)"
            after = {"CDF": score_dataset(model, rest), "FF++": score_dataset(model, DATA["FF++"]),
                     "WDF": score_dataset(model, DATA["WDF"])}
            b_cdf = BASE[name]["CDF"][BASE[name]["CDF"].uid.isin(set(rest.uid))]
            before = {"CDF": b_cdf, "FF++": BASE[name]["FF++"], "WDF": BASE[name]["WDF"]}
            for ds in DATASETS:
                ROWS.append({"model": name, "variant": name.rsplit("_s", 1)[0], "train_seed": int(name.rsplit("_s", 1)[1]),
                             "k": k, "draw": d, "dataset": ds, "n": len(after[ds]),
                             "AUC_before": auc(before[ds]), "AUC_after": auc(after[ds]),
                             "loss_first": losses[0], "loss_last": losses[-1]})
            r_ = [x for x in ROWS[-3:]]
            print(f"  {k}-shot draw {d}: " + " · ".join(f"{x['dataset']} {x['AUC_before']:.2f}→{x['AUC_after']:.2f}" for x in r_)
                  + f"  (loss {losses[0]:.3f}→{losses[-1]:.3f})")
    del model; torch.cuda.empty_cache()
    print(f"  {(time.time() - t0) / 60:.1f} นาที")

R = pd.DataFrame(ROWS); R["ΔAUC"] = R.AUC_after - R.AUC_before
R.to_csv(OUT_DIR / "fewshot_runs.csv", index=False)

## 7. สรุป: mean ± SD (ข้าม train seed × การสุ่มชุดภาพ)

- **CDF** = ชุดเป้าหมาย (วัดบนวิดีโอที่เหลือ · identity ไม่ซ้ำกับภาพที่ใช้ปรับ) → ปรับตัวได้ไหม
- **FF++** = ชุดเดิม → ลืมของเดิมไหม (บทความ: FB/CFB "maintains stable performance in the source dataset")
- **WDF** = ไม่เคยเห็นทั้งตอนฝึกและตอนปรับ

In [ ]:
S_ = (R.groupby(["variant", "k", "dataset"])
       .agg(before=("AUC_before", "mean"), after=("AUC_after", "mean"),
            delta=("ΔAUC", "mean"), delta_sd=("ΔAUC", "std"), runs=("ΔAUC", "size"))
       .round(2).reset_index())
S_.to_csv(OUT_DIR / "fewshot_summary.csv", index=False)
print("ΔAUC = หลังปรับ − ก่อนปรับ (จุด) · mean ± SD ข้าม train seed × draw")
for ds in ["CDF", "FF++", "WDF"]:
    print(f"\n=== {ds} ===")
    print(S_[S_.dataset == ds].to_string(index=False))

# paired: CFB full vs FB·CE ใช้ชุดภาพเดียวกัน (k, draw) และ train seed เดียวกัน
P = R.pivot_table(index=["train_seed", "k", "draw", "dataset"], columns="variant", values="ΔAUC").dropna()
if {"fb_ce", "cfb_full"} <= set(P.columns):
    P["diff"] = P["cfb_full"] - P["fb_ce"]
    G = P.groupby(["k", "dataset"])["diff"].agg(["mean", "std", "count"]).round(2)
    print("\nΔ(CFB full) − Δ(FB·CE) · paired ตามชุดภาพและ train seed (> 0 = CFB ปรับตัวได้ดีกว่า)")
    print(G.to_string())
    G.to_csv(OUT_DIR / "fewshot_cfb_vs_fb.csv")
print("\nบทความ Table V (CFB, AUC ก่อน → หลัง):", PAPER_TABLE5)

In [ ]:
INK2, GRID = "#52514e", "#e6e5e0"
COL = {"fb_ce": "#b7b0a7", "cfb_full": "#7a1f22"}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
for ax, ds in zip(axes, ["CDF", "FF++", "WDF"]):
    for j, v in enumerate(["fb_ce", "cfb_full"]):
        for i_, k in enumerate(SHOTS):
            d = R[(R.dataset == ds) & (R.variant == v) & (R.k == k)]["ΔAUC"].values
            if not len(d): continue
            x = i_ + (j - 0.5) * 0.28
            ax.scatter(np.full(len(d), x) + np.random.uniform(-0.04, 0.04, len(d)), d, s=22, color=COL[v], alpha=0.7)
            ax.hlines(d.mean(), x - 0.11, x + 0.11, color=COL[v], lw=3)
    ax.axhline(0, color=INK2, lw=1)
    ax.set_xticks(range(len(SHOTS))); ax.set_xticklabels([f"{k}-shot" for k in SHOTS])
    ax.set_title({"CDF": "Celeb-DF (เป้าหมาย)", "FF++": "FF++ (ชุดเดิม)", "WDF": "WildDeepfake (ไม่เคยเห็น)"}[ds], fontsize=11)
    ax.grid(axis="y", color=GRID); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("ΔAUC หลัง − ก่อน (จุด)")
from matplotlib.lines import Line2D
fig.legend([Line2D([], [], color=COL[v], marker="o", lw=3) for v in COL], ["FB · CE", "CFB full"],
           loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(OUT_DIR / "fewshot_delta.png", dpi=160); plt.show()
print("ไฟล์ผลลัพธ์:", sorted(p.name for p in OUT_DIR.iterdir() if not p.name.startswith("scores_")))
print(f"เวลารวม: {(time.time() - T_START) / 60:.1f} นาที")